# 02 — Prepare Dataset

**Fruvia AI** — Dataset preparation notebook.

This notebook runs on **Google Colab** and produces two manifest CSV files:

1. **`retrieval_manifest.csv`** — Every valid image in the dataset (used for DINOv2 embedding)
2. **`classification_manifest.csv`** — Only images mapped to the 18 target classes,
   with stratified train/validation/test splits (70/15/15, seed=42)

### What this notebook does

1. Load Fruits-360 dataset (download if needed)
2. Clone the Fruvia AI repo to access `classes.yaml` and `class_mapping.yaml`
3. Validate every image with Pillow
4. Compute SHA-256 hashes for duplicate detection
5. Build `retrieval_manifest.csv` (all valid images)
6. Build `classification_manifest.csv` (target classes only, deduplicated, stratified split)
7. Print pre/post statistics
8. Save manifests to Google Drive

### Prerequisites

- Kaggle credentials in Colab Secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`)
- No GPU required — CPU runtime is sufficient

## 1. Setup & Dependencies

In [ ]:
!pip install -q kagglehub pyyaml Pillow pandas

In [ ]:
import hashlib
import os
import random
import uuid
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import pandas as pd
import yaml
from PIL import Image

random.seed(42)

## 2. Download Dataset & Clone Config

In [ ]:
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import kagglehub

dataset_path = kagglehub.dataset_download("moltean/fruits")
DATA_DIR = Path(dataset_path)
print(f"Dataset at: {DATA_DIR}")
print(f"Contents: {sorted([p.name for p in DATA_DIR.iterdir()])}")

In [ ]:
# Clone the repo to get config files
REPO_DIR = Path("/content/fruvia-ai")
if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/dinhvien04/fruvia-ai.git /content/fruvia-ai
else:
    print(f"Repo already cloned at {REPO_DIR}")

CLASSES_YAML = REPO_DIR / "configs" / "classes.yaml"
MAPPING_YAML = REPO_DIR / "configs" / "class_mapping.yaml"

assert CLASSES_YAML.exists(), f"Missing {CLASSES_YAML}"
assert MAPPING_YAML.exists(), f"Missing {MAPPING_YAML}"

## 3. Load Config Files

In [ ]:
with open(CLASSES_YAML, encoding="utf-8") as f:
    classes_cfg = yaml.safe_load(f)
TARGET_CLASSES: list[str] = classes_cfg["classes"]

with open(MAPPING_YAML, encoding="utf-8") as f:
    mapping_cfg = yaml.safe_load(f)
CLASS_MAPPING: dict[str, str] = mapping_cfg["class_mapping"]

print(f"Target classes ({len(TARGET_CLASSES)}): {TARGET_CLASSES}")
print(f"Class mappings: {len(CLASS_MAPPING)} original → {len(TARGET_CLASSES)} target")

## 4. Utility Functions

In [ ]:
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}
UUID_NAMESPACE = uuid.UUID("a3f1b2c4-d5e6-4f7a-8b9c-0d1e2f3a4b5c")


def compute_sha256(filepath: Path, chunk_size: int = 65536) -> str:
    """Compute SHA-256 hex digest of a file."""
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def generate_image_id(relative_path: str) -> str:
    """Deterministic UUID5 from relative path."""
    return str(uuid.uuid5(UUID_NAMESPACE, relative_path))


def validate_image(filepath: Path) -> tuple[bool, tuple[int, int] | None]:
    """Check if Pillow can open and verify the image."""
    try:
        with Image.open(filepath) as img:
            img.verify()
        with Image.open(filepath) as img:
            return True, img.size
    except Exception:
        return False, None

## 5. Scan All Images

In [ ]:
def scan_dataset(data_dir: Path) -> list[dict[str, Any]]:
    """Scan dataset directory and return metadata for every image."""
    records = []
    for filepath in sorted(data_dir.rglob("*")):
        if not filepath.is_file():
            continue
        if filepath.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        relative_path = str(filepath.relative_to(data_dir))
        original_class = filepath.parent.name

        # Determine source split from directory structure
        parts = Path(relative_path).parts
        source = parts[0] if len(parts) >= 3 else "unknown"

        # Validate
        is_valid, size = validate_image(filepath)
        width, height = size if size else (None, None)

        # Hash
        try:
            sha256 = compute_sha256(filepath)
        except OSError:
            sha256 = None
            is_valid = False

        records.append(
            {
                "image_id": generate_image_id(relative_path),
                "original_class": original_class,
                "relative_path": relative_path,
                "filename": filepath.name,
                "width": width,
                "height": height,
                "file_size": filepath.stat().st_size,
                "sha256": sha256,
                "source": source,
                "is_valid": is_valid,
            }
        )
    return records


print("Scanning dataset (this may take a few minutes)...")
all_records = scan_dataset(DATA_DIR)
print(f"Total files scanned: {len(all_records)}")
print(f"Valid images: {sum(1 for r in all_records if r['is_valid'])}")
print(f"Invalid images: {sum(1 for r in all_records if not r['is_valid'])}")

## 6. Create Retrieval Manifest (All Images)

In [ ]:
# Retrieval manifest: ALL valid images — used for DINOv2 embedding
retrieval_records = [r for r in all_records if r["is_valid"]]

# Add target_class (if mapped) and split="gallery" for retrieval
for r in retrieval_records:
    r["target_class"] = CLASS_MAPPING.get(r["original_class"], "unmapped")
    r["split"] = "gallery"  # All images go to gallery for retrieval

# Deduplicate by SHA-256
seen_hashes: set[str] = set()
retrieval_unique: list[dict] = []
retrieval_dups = 0
for r in retrieval_records:
    if r["sha256"] and r["sha256"] in seen_hashes:
        retrieval_dups += 1
        continue
    if r["sha256"]:
        seen_hashes.add(r["sha256"])
    retrieval_unique.append(r)

print(
    f"Retrieval manifest: {len(retrieval_unique)} unique images ({retrieval_dups} duplicates removed)"
)

# Save
RETRIEVAL_CSV = Path("/content/retrieval_manifest.csv")
MANIFEST_COLUMNS = [
    "image_id",
    "original_class",
    "target_class",
    "relative_path",
    "filename",
    "width",
    "height",
    "file_size",
    "sha256",
    "split",
    "source",
    "is_valid",
]

df_retrieval = pd.DataFrame(retrieval_unique)
df_retrieval[MANIFEST_COLUMNS].to_csv(RETRIEVAL_CSV, index=False)
print(f"Saved to {RETRIEVAL_CSV}")

## 7. Create Classification Manifest (Target Classes Only)

In [ ]:
# Classification manifest: only mapped target classes, stratified split
target_set = set(TARGET_CLASSES)

# Filter to valid, mapped, target-class images
clf_records = []
skipped_unmapped = 0
skipped_not_target = 0

for r in all_records:
    if not r["is_valid"]:
        continue
    target = CLASS_MAPPING.get(r["original_class"])
    if target is None:
        skipped_unmapped += 1
        continue
    if target not in target_set:
        skipped_not_target += 1
        continue
    rec = dict(r)
    rec["target_class"] = target
    clf_records.append(rec)

print(f"Classification candidates: {len(clf_records)}")
print(f"Skipped (unmapped): {skipped_unmapped}")
print(f"Skipped (not target): {skipped_not_target}")

# Deduplicate by SHA-256
seen_hashes = set()
clf_unique = []
clf_dups = 0
for r in clf_records:
    if r["sha256"] and r["sha256"] in seen_hashes:
        clf_dups += 1
        continue
    if r["sha256"]:
        seen_hashes.add(r["sha256"])
    clf_unique.append(r)

print(f"After dedup: {len(clf_unique)} ({clf_dups} duplicates removed)")

In [ ]:
# Stratified split: 70% train / 15% validation / 15% test
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
SEED = 42

rng = random.Random(SEED)

# Group by target class
by_class: dict[str, list[dict]] = defaultdict(list)
for r in clf_unique:
    by_class[r["target_class"]].append(r)

split_records: list[dict] = []
for cls in sorted(by_class.keys()):
    items = by_class[cls]
    rng.shuffle(items)
    # Sort by SHA-256 to cluster near-duplicates in same split
    items.sort(key=lambda x: x.get("sha256", ""))

    n = len(items)
    n_train = max(1, int(n * TRAIN_RATIO))
    n_val = max(1, int(n * VAL_RATIO))

    for i, item in enumerate(items):
        if i < n_train:
            item["split"] = "train"
        elif i < n_train + n_val:
            item["split"] = "validation"
        else:
            item["split"] = "test"
        split_records.append(item)

# Stats
split_counts = Counter(r["split"] for r in split_records)
print("\nClassification manifest split distribution:")
for split, count in sorted(split_counts.items()):
    pct = count / len(split_records) * 100
    print(f"  {split:15s}: {count:6d} ({pct:.1f}%)")
print(f"  {'TOTAL':15s}: {len(split_records):6d}")

# Save
CLF_CSV = Path("/content/classification_manifest.csv")
df_clf = pd.DataFrame(split_records)
df_clf[MANIFEST_COLUMNS].to_csv(CLF_CSV, index=False)
print(f"\nSaved to {CLF_CSV}")

## 8. Pre/Post Statistics

In [ ]:
print("=" * 60)
print("DATASET PREPARATION SUMMARY")
print("=" * 60)
print(f"  Raw images scanned:           {len(all_records)}")
print(f"  Valid images:                  {sum(1 for r in all_records if r['is_valid'])}")
print(f"  Invalid / corrupt:             {sum(1 for r in all_records if not r['is_valid'])}")
print(f"  Unique original classes:       {len(set(r['original_class'] for r in all_records))}")
print()
print(f"  Retrieval manifest:            {len(retrieval_unique)} images (all classes)")
print(
    f"  Classification manifest:       {len(split_records)} images ({len(TARGET_CLASSES)} target classes)"
)
print(f"    Duplicates removed (retr.):  {retrieval_dups}")
print(f"    Duplicates removed (clf.):   {clf_dups}")
print()
print("Per-class distribution (classification):")
class_counts = Counter(r["target_class"] for r in split_records)
for cls, count in sorted(class_counts.items()):
    print(f"  {cls:<20s}: {count:>6d}")
print("=" * 60)

## 9. (Optional) Save Manifests to Google Drive

In [ ]:
# Uncomment to save manifests to Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
#
# import shutil
# DRIVE_DIR = Path("/content/drive/MyDrive/fruvia-ai/manifests")
# DRIVE_DIR.mkdir(parents=True, exist_ok=True)
#
# shutil.copy2(RETRIEVAL_CSV, DRIVE_DIR / RETRIEVAL_CSV.name)
# shutil.copy2(CLF_CSV, DRIVE_DIR / CLF_CSV.name)
# print(f"Manifests saved to {DRIVE_DIR}")

---

## Summary

| Output | Description |
|--------|-------------|
| `retrieval_manifest.csv` | All valid unique images for DINOv2 embedding |
| `classification_manifest.csv` | Target-class images, stratified 70/15/15 split |

**Next step:** Use manifests for training (notebooks 03–05) and embedding (notebook 06).